In [ ]:
!pip install -q torch torchaudio soundfile datasets

In [ ]:
!pip install -q torch torchaudio soundfile datasets

import os
import torch
import torchaudio
import soundfile as sf
from datasets import load_dataset

print("Torch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)

In [ ]:
from datasets import load_dataset

ds_train = load_dataset("szzs1693/coswara-data", "audio", split="train")
ds_val   = load_dataset("szzs1693/coswara-data", "audio", split="val")
ds_test  = load_dataset("szzs1693/coswara-data", "audio", split="test")

print("Loaded:", len(ds_train), len(ds_val), len(ds_test))

In [ ]:
unique_audio_types = sorted(set(ds_train["audio_type"]))
print(unique_audio_types)

In [ ]:
cough_types = ["cough-heavy", "cough-shallow"]

ds_cough = ds_train.filter(lambda x: x["audio_type"] in cough_types)

print("Total cough samples:", len(ds_cough))

# double check
print("Unique types in filtered dataset:")
print(sorted(set(ds_cough["audio_type"])))

In [ ]:
import torch
import torchaudio

bundle = torchaudio.pipelines.SQUIM_OBJECTIVE
device = "cuda" if torch.cuda.is_available() else "cpu"

model = bundle.get_model().to(device)
model.eval()

print("SQUIM loaded on:", device)

In [ ]:
THRESHOLD_STOI = 0.6
THRESHOLD_PESQ = 1.15
THRESHOLD_SI_SDR = -12

#use only 1000 samples for testing
ds_cough_small = ds_cough.select(range(min(1000, len(ds_cough))))
print("Subset size:", len(ds_cough_small))

good_files = []
bad_files = []

for i, example in enumerate(ds_cough_small):

    audio_data = example["audio"]

    # skip broken samples
    if audio_data is None or audio_data["array"] is None:
        continue

    waveform = audio_data["array"]
    sr = audio_data["sampling_rate"]

    waveform_tensor = torch.tensor(waveform).float()

    # convert to mono
    if waveform_tensor.ndim == 1:
        waveform_tensor = waveform_tensor.unsqueeze(0)
    elif waveform_tensor.ndim > 1 and waveform_tensor.shape[0] > 1:
        waveform_tensor = waveform_tensor.mean(dim=0, keepdim=True)

    # resample to 16kHz
    waveform_tensor = torchaudio.functional.resample(waveform_tensor, sr, 16000)
    waveform_tensor = waveform_tensor.to(device)

    # run SQUIM
    with torch.no_grad():
        stoi, pesq, si_sdr = model(waveform_tensor)

    # classify
    if (
        stoi.item() >= THRESHOLD_STOI
        and pesq.item() >= THRESHOLD_PESQ
        and si_sdr.item() >= THRESHOLD_SI_SDR
    ):
        good_files.append(example)
    else:
        bad_files.append(example)

print("✅ Good cough files:", len(good_files))
print("❌ Bad cough files:", len(bad_files))

In [ ]:
import os
import pandas as pd
import soundfile as sf

# ✅ thresholds (cough-adapted)
THRESHOLD_STOI = 0.6
THRESHOLD_PESQ = 1.15
THRESHOLD_SI_SDR = -12

# ✅ folders
os.makedirs("good_coughs", exist_ok=True)
os.makedirs("bad_coughs", exist_ok=True)

results = []

# ✅ use 1000 samples (change to ds_cough for full dataset)
ds_cough_subset = ds_cough.select(range(min(1000, len(ds_cough))))
print("Processing samples:", len(ds_cough_subset))

for i, example in enumerate(ds_cough_subset):

    audio_data = example["audio"]

    if audio_data is None or audio_data["array"] is None:
        continue

    waveform = audio_data["array"]
    sr = audio_data["sampling_rate"]

    waveform_tensor = torch.tensor(waveform).float()

    # mono
    if waveform_tensor.ndim == 1:
        waveform_tensor = waveform_tensor.unsqueeze(0)
    elif waveform_tensor.ndim > 1 and waveform_tensor.shape[0] > 1:
        waveform_tensor = waveform_tensor.mean(dim=0, keepdim=True)

    # resample
    waveform_tensor = torchaudio.functional.resample(waveform_tensor, sr, 16000)
    waveform_tensor = waveform_tensor.to(device)

    # SQUIM inference
    with torch.no_grad():
        stoi, pesq, si_sdr = model(waveform_tensor)

    stoi_val = stoi.item()
    pesq_val = pesq.item()
    si_sdr_val = si_sdr.item()

    # classification
    label = "good" if (
        stoi_val >= THRESHOLD_STOI and
        pesq_val >= THRESHOLD_PESQ and
        si_sdr_val >= THRESHOLD_SI_SDR
    ) else "bad"

    # save files
    if label == "good":
        sf.write(f"good_coughs/good_cough_{i}.wav", waveform, sr)
    else:
        sf.write(f"bad_coughs/bad_cough_{i}.wav", waveform, sr)

    # store results
    results.append({
        "index": i,
        "audio_type": example["audio_type"],
        "STOI": stoi_val,
        "PESQ": pesq_val,
        "SI-SDR": si_sdr_val,
        "label": label
    })

# ✅ dataframe
df_results = pd.DataFrame(results)

# ✅ save CSV
df_results.to_csv("cough_quality_results.csv", index=False)

# ✅ summary
print("\n=== Summary ===")
print("Total processed:", len(df_results))
print(df_results["label"].value_counts())

print("\n=== Averages ===")
print(df_results.groupby("label")[["STOI", "PESQ", "SI-SDR"]].mean())